# Self-Evolving Agents (MASE) | Frameworks & Meta-Approaches

In [1]:
# Self-Evolving Agent (MASE): Intent Classifier with Sandbox Validation
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# --- Test set: 5 queries with expected intent labels ---
TEST_SET = [
    ("I want my money back for order #442",          "refund"),
    ("How do I reset my password?",                   "account_access"),
    ("Your app keeps crashing on Android",            "bug_report"),
    ("Can I upgrade to the business plan?",           "upgrade"),
    ("I was charged twice this month",                "billing"),
]
KEYWORDS = {  # required keywords in response for each label
    "refund": ["refund"], "account_access": ["password", "reset"],
    "bug_report": ["bug", "crash"], "upgrade": ["upgrade", "plan"],
    "billing": ["billing", "charge"],
}

def evaluate(prompt: str) -> tuple:
    """Run prompt on all 5 test queries; return (score, failure_details)."""
    correct, failures = 0, []
    for query, expected in TEST_SET:
        resp = model.invoke(f"{prompt}\n\nClassify this query: {query}").content.lower()
        if any(kw in resp for kw in KEYWORDS[expected]):
            correct += 1
        else:
            failures.append(f"Query: '{query}' | Expected: {expected} | Got: {resp[:80]}")
    score = correct / len(TEST_SET)
    return score, failures

def evolve_prompt(current_prompt: str, failures: list) -> str:
    """Ask the LLM to improve the prompt based on failure analysis."""
    failure_text = "\n".join(failures)
    resp = model.invoke(
        f"You are improving a customer intent classifier.\n"
        f"Current system prompt:\n{current_prompt}\n\n"
        f"These queries were misclassified:\n{failure_text}\n\n"
        f"The valid intent labels are: refund, account_access, bug_report, upgrade, billing.\n"
        f"Write an improved system prompt that classifies more accurately. "
        f"Return ONLY the new prompt, nothing else."
    )
    return resp.content

In [4]:
# --- Evolution loop with sandbox validation ---
prompt = "You are a customer service assistant. Respond to the user's query."
best_score, _ = evaluate(prompt)
print(f"v1 | Baseline score: {best_score:.0%}")

for cycle in range(1, 4):
    # Evaluate current prompt to find failures
    current_score, failures = evaluate(prompt)
    if not failures:
        print(f"  Perfect score -- stopping early.")
        break

    # Generate candidate prompt
    candidate = evolve_prompt(prompt, failures)

    # SANDBOX TEST: run candidate on the SAME 5 queries
    candidate_score, _ = evaluate(candidate)
    print(f"v{cycle+1} | Current: {current_score:.0%} | Candidate: {candidate_score:.0%}", end=" ")

    if candidate_score > best_score:
        prompt, best_score = candidate, candidate_score
        print("-> PROMOTED")
    else:
        print("-> REJECTED (keeping old prompt)")

print(f"\nFinal score: {best_score:.0%}")
print(f"Final prompt: {prompt[:200]}...")

v1 | Baseline score: 100%
  Perfect score -- stopping early.

Final score: 100%
Final prompt: You are a customer service assistant. Respond to the user's query....
